# Molecular Design VAE — Large CPU Mode

**~100,000 ZINC molecules · 400 epochs · CPU · ~4-6 hours**

**WARNING:** Free Google Colab will likely time out before this finishes training. This notebook does not start a live UI server — it shows results directly in the notebook output cells.

If you want a live UI, use **Large GPU mode** instead (`colab_large_gpu.ipynb`), which finishes in 20-40 minutes.

**To actually use Large CPU mode**, run it locally overnight, not on Colab.

## Cell 1 — Setup

In [ ]:
!git clone https://github.com/Kaur-Simarpreet/molecular-design-vae.git
%cd molecular-design-vae
!pip install -q torch selfies flask flask-cors scipy requests
!pip install -q rdkit
print('Setup complete')
print('NOTE: Large CPU mode trains for 4-6 hours — Colab will likely time out.')

## Cell 1b — Set up real Vina docking (~2 min)

Optional but recommended for Large mode. Without this, scoring uses mock RDKit estimates.

In [ ]:
!bash setup_docking.sh --vina
!python docking.py   # verify


## Cell 2 — Train (will probably time out — saves best checkpoint as it goes)

In [ ]:
!python train_vae_extended.py --mode large

## Cell 3 — Use whatever was trained (best checkpoint)

Even if Cell 2 was interrupted, `vae_best.pt` holds the best validity achieved so far. This cell loads it and generates molecules directly in the notebook.

In [ ]:
import shutil, os
if os.path.exists('saved_model/vae_best.pt') and not os.path.exists('saved_model/vae.pt'):
    shutil.copy('saved_model/vae_best.pt', 'saved_model/vae.pt')
    print('Using vae_best.pt (best validity from interrupted training)')

# Generate sample molecules in-process — no server needed
import sys
sys.path.insert(0, '.')
from serve import vae, tokenizer, score_mol
import torch, random

weights = {'qed':0.30, 'dock':0.40, 'sa':0.15, 'nov':0.05, 'admet':0.10}
generated = []
for _ in range(30):
    z = torch.randn(1, vae.encoder.mu.out_features)
    smi = vae.decode_z(z, tokenizer, temperature=0.9)
    if smi:
        scored = score_mol(smi, 'EGFR kinase', weights=weights)
        if scored:
            generated.append(scored)

print(f'Generated {len(generated)} molecules')

## Cell 4 — Display table and export CSV

In [ ]:
import pandas as pd

df = pd.DataFrame(generated)
cols = ['smiles', 'qed', 'sa_score', 'docking_score', 'rl_reward', 'novelty']
available = [c for c in cols if c in df.columns]
print(df[available].to_string())

df.to_csv('large_mode_results.csv', index=False)
print('\nResults saved to large_mode_results.csv')

## Cell 5 — Download results

In [ ]:
from google.colab import files
files.download('large_mode_results.csv')
import shutil
shutil.make_archive('saved_model', 'zip', 'saved_model')
files.download('saved_model.zip')